In [ ]:
import os, sys
sys.path.insert(0, str(__import__('pathlib').Path.cwd().parent))

from src.config_validation import load_and_validate_config, require

EXPERIMENT_CONFIG = load_and_validate_config('NB00')

DATASET_NAME = require(EXPERIMENT_CONFIG, 'dataset', 'name')
DATASET_CONFIG_FILE = require(EXPERIMENT_CONFIG, 'dataset', 'config_file')

print(f'Dataset: {DATASET_NAME}')
print(f'Config:  {DATASET_CONFIG_FILE}')

In [ ]:
import yaml
from pathlib import Path
from src.datasets import load_dataset, list_datasets, get_dataset_info

print('Available datasets:', list_datasets())
print()

for ds_name in list_datasets():
    info = get_dataset_info(ds_name)
    print(f"--- {info['name']} ---")
    print(f"  Description: {info['description']}")
    print(f"  Conditions:  {info.get('conditions', 'N/A')}")
    print(f"  Format:      {info.get('format', 'N/A')}")
    print()

In [ ]:
# Load the selected dataset
ds_config_path = Path('..') / DATASET_CONFIG_FILE
if not ds_config_path.exists():
    raise FileNotFoundError(f'Dataset config not found: {ds_config_path}')

with open(ds_config_path) as f:
    ds_config = yaml.safe_load(f)

dataset = load_dataset(DATASET_NAME, ds_config)
print(dataset)
print()
print('Dataset info:')
for k, v in dataset.info().items():
    print(f'  {k}: {v}')

In [ ]:
# List conditions and sample counts
conditions = dataset.list_available_conditions()
print(f'Available conditions ({len(conditions)}):')
print()

for condition in conditions:
    try:
        sample_ids, labels = dataset.get_disease_labels(condition)
        n_case = int(labels.sum())
        n_ctrl = len(labels) - n_case
        print(f'  {condition:25s}  cases={n_case:5d}  controls={n_ctrl:5d}  total={len(labels):5d}')
    except Exception as e:
        print(f'  {condition:25s}  ERROR: {e}')

In [ ]:
# Check phylogeny availability
phylo = dataset.load_phylogeny()
if phylo is not None:
    print('Phylogenetic tree: AVAILABLE')
    print(f'  Graph construction will use phylogenetic distance.')
else:
    print('Phylogenetic tree: NOT AVAILABLE')
    print(f'  Graph construction will use k-NN on abundance similarity.')